In [ ]:
import shutil
import os

if os.path.exists("logs"):
    shutil.rmtree("logs")
    print("Logs directory cleared")
else:
    print("No logs directory found")

In [ ]:
from utils import start_tensorboard

start_tensorboard()

# Convolutional Neural Networks: Image Classification with CNNs
Moving beyond fully connected networks, we explore Convolutional Neural Networks (CNNs) designed specifically for image data. CNNs preserve spatial structure through convolutional layers, pooling operations, and hierarchical feature learning.

In [ ]:
# load data
import keras
import numpy as np


(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()


X_train_scaled = X_train.astype("float32") / 255.0
X_test_scaled = X_test.astype("float32") / 255.0


y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"Classes: {np.unique(y_train)}")

## The issues with fully connected dense networks for images
As previously discussed, dense networks have a few major downsides as we increase image size and scene complexity. I want you to take a look at what happens to the number of parameters of a dense NN when using the **standard** image size of many more complex datasets, 224x224 pixels. (Quite small by modern standards, right?)

So please everyone build a non-functional model consisting of just an input layer with the flattened dimensions of 224*224 and a single dense layer of 256 neurons. (Tell me what you see!)

Then think about what happens with convolutional (2D) layers. Does the number of parameters grow in the same way? What about color images?

In [ ]:
from keras.layers import Input, Dense
from keras.models import Model
inputs = Input((224*224,))
x = Dense(256, activation='relu')(inputs)
x = Dense(128, activation='relu')(x)
model = Model(inputs=inputs, outputs=x)
model.summary()

## Whats a Convolutional Layer? Whats a Filter?
Some of you may have seen image filters like the Sobel filter already. A (often quadratic) 2d filter is defined. Then this filter slides over the image, where the output is the sum of products of each filter position and its corresponding image position (think dot product!). The thing that makes a Conv2D layer different from a classic filter, is that the filter values are *learned* (our trainable weights). 


We simply define the shape and number of filters in a layer but the exact filter values are learned! Over the course of multiple Conv2D layers, we then expect the layers to learn more complex shapes and patterns to look for in the image, because the receptive field increases. Q: Whats the receptive field?

One thing that is often lost, is that the Convolutional2D layer is not 2d, because it also depends on the number of input channels, e.g. RGB images -> Each filter is how big if 3x3 filters are used?

Each filter also has a bias term!


## Calculate by Hand: CNN Architecture
Lets take a look at a small convolutional neural network and how to think about parameters, as well as input and output shapes for this.

**Task:** Calculate parameters and output shapes for each layer. Does input size affect parameter count?

**Conv2D parameters:** `(kernel_h × kernel_w × input_channels + 1) × filters`

**Output shape:** Depends on kernel size, stride, padding (`same` preserves size, `valid` shrinks).

**Key:** What about different image sizes for inputs? How does the number of parameters change?

| Layer | Input Shape | Filters (C_out) | Kernel Size | Stride | Padding | Output Shape = ? | Params = ? |
|-------|-------------|-----------------|-------------|--------|---------|------------------|------------|
| Conv1 | 32 × 32 × 3 | 8 | 3 × 3 | 1 | same | ? | ? |
| Conv2 | (output of Conv1) | 16 | 3 × 3 | 1 | valid | ? | ? |
| Conv3 | (output of Conv2) | 4 | 5 × 5 | 1 | valid | ? | ? |

In [ ]:
from keras.layers import Conv2D

#what about inpout size of images now? does it matter in terms of paramters of our network?
inputs = Input(shape=(32, 32, 3))
x = Conv2D(filters=8, kernel_size=(3, 3), strides=1, padding="same", name="conv1")(inputs)
x = Conv2D(filters=16, kernel_size=(3, 3), strides=1, padding="valid", name="conv3")(x)
outputs = Conv2D(filters=4, kernel_size=(5, 5), strides=1, padding="valid", name="conv4")(x)
# what exactly is the output of this
model_ = Model(inputs=inputs, outputs=outputs)
model_.summary()

## Add Classification Head
This is fine and all, but how do we actually get this to output probabilities of our classes we want to predict?
For this we introduce a new layer type:
### Flatten Layer
Converts 2D feature maps (height × width × channels) to 1D vector for Dense layers. Example: (24 Height, 24 Width, 64 Feature Maps) → (36864,)= a long 1d feature array. No trainable parameters.
We can then do a typical dense layer to down project into more compact feature representations and finally use the same dense output layer as last time with the fcnn.

What do you notice in terms of parameters now for our final model?


In [ ]:
from keras.layers import Conv2D, Flatten, Dense

inputs = Input(shape=(28, 28, 1))
x = Conv2D(32, (3, 3), activation='relu')(inputs)
x = Conv2D(64, (3, 3), activation='relu')(x)

#flatten the final feature maps into a 1d feature vector/array
x = Flatten()(x)
x = Dense(128, activation='relu')(x) # down projection
outputs = Dense(10, activation='softmax')(x)  # 10 classes

model_ = Model(inputs=inputs, outputs=outputs)
model_.summary()

### MaxPooling Layer

Reduces spatial dimensions by taking maximum value in each pooling window. Decreases computation, adds **some minor** translation invariance, controls overfitting.

**Key parameters:**
- `pool_size`: Window size (e.g., (2,2))
- `strides`: Default = pool_size
- `padding`: Usually "valid"

**Effect:** (28, 28, 32) with pool_size=(2,2) → (14, 14, 32). Channels unchanged, spatial dims halved. No trainable parameters.

Question: What happens to the parameter count? Where does the big effect come from?


In [ ]:
from keras.layers import MaxPooling2D, Dropout, GlobalAveragePooling2D

# TODO: Build and compile a CNN for CIFAR-10.
# Requirements:
# - input shape: (32, 32, 3)
# - several Conv2D layers for feature extraction
# - downsampling, for example with MaxPooling2D
# - a classification head that outputs 10 class probabilities
# - compile the model with a suitable optimizer, loss, and accuracy metric
#
# Store the final compiled model in the variable `model`, so the next cells can train it.
raise NotImplementedError("Build the CIFAR-10 CNN and assign it to `model`.")

In [ ]:
# train and eval on test set
from utils import train_model, eval_classification

basic_cnn_history = train_model(model, X_train_scaled, y_train, "basic_Conv_Net_CIFAR10", epochs=10)
eval_classification(model, X_test_scaled, y_test, class_names, "CIFAR-10 CNN")

## Loading Images from Directories

For datasets that are too large to fit into memory, keep the images on disk in one folder per class and let Keras stream batches from those folders.

CIFAR-10 is small, so here we first write it into the same directory layout: `train/class_name/image.png`, `validation/class_name/image.png`, and `test/class_name/image.png`.

In [ ]:
import shutil
from pathlib import Path
from keras.utils import save_img

cifar_dir = Path("data/cifar10_images")
validation_fraction = 0.1
rng = np.random.default_rng(42)

train_indices = []
validation_indices = []
for class_id in np.unique(y_train):
    class_indices = np.where(y_train == class_id)[0]
    rng.shuffle(class_indices)
    n_validation = int(len(class_indices) * validation_fraction)
    validation_indices.extend(class_indices[:n_validation])
    train_indices.extend(class_indices[n_validation:])

splits = {
    "train": (X_train[train_indices], y_train[train_indices]),
    "validation": (X_train[validation_indices], y_train[validation_indices]),
    "test": (X_test, y_test),
}

if cifar_dir.exists():
    shutil.rmtree(cifar_dir)


for split_name in splits: # forr train, val and test
    for class_name in class_names: # for airplane, truck, car ...
        (cifar_dir / split_name / class_name).mkdir(parents=True, exist_ok=True) # make a folder to save the images

for split_name, (images, labels) in splits.items():
    for i, (image, label) in enumerate(zip(images, labels)):
        image_path = cifar_dir / split_name / class_names[int(label)] / f"{i:05d}.png" # eg cifar10/train/airplane/00001.png
        save_img(image_path, image)

print(f"CIFAR-10 directory dataset ready at: {cifar_dir}")

In [ ]:
import tensorflow as tf

batch_size = 64
img_size = (32, 32)

train_ds = keras.utils.image_dataset_from_directory(
    cifar_dir / "train",
    class_names=class_names,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=True,
    seed=42,
)

validation_ds = keras.utils.image_dataset_from_directory(
    cifar_dir / "validation",
    class_names=class_names,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
)

test_ds = keras.utils.image_dataset_from_directory(
    cifar_dir / "test",
    class_names=class_names,
    image_size=img_size,
    batch_size=batch_size,
    shuffle=False,
)

rescale = keras.layers.Rescaling(1.0 / 255)
train_ds = train_ds.map(lambda images, labels: (rescale(images), labels)).prefetch(tf.data.AUTOTUNE) # prefetch the next batch WHILE we are training on the current one so we use CPU and GPU
validation_ds = validation_ds.map(lambda images, labels: (rescale(images), labels)).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda images, labels: (rescale(images), labels)).prefetch(tf.data.AUTOTUNE)

## Data Augmentation

For most modern networks that are actually used in production, architecture is not the defining factor - high quality data, and how well it captures the real world problem, is. Often the process of sampling data, getting images etc. is expensive and we cant always have the amount of data we need to train a NN well. 

One mitigation in computer vision is a set data augmentation strategies that mostly consist of **simple affine tranformations** of our input images. This is often:
- Random Crop, Flip, Zoom
- Random Translation (shifting)
- Color or Brightness Jitter

**The cool thing is that we apply those tranformations dynamically to each new batch randomly.**

Lets look at how keras does this with a set of augmentation layers.

NOTE: When we choose augmentations we should always think about what makes sense in a real world setting - which augmentations could exist when we use the model later for prediction? (e.g. compare Horizontal vs Vertical Flip) 

https://keras.io/api/layers/preprocessing_layers/

In [ ]:
from keras.layers import RandomFlip, RandomRotation, RandomZoom, RandomTranslation

data_augmentation = keras.Sequential([
    ##TODO Use various agumentation layer
])

sample_images, sample_labels = next(iter(train_ds))
augmented_images = data_augmentation(sample_images[:12], training=True)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 6, figsize=(12, 6))
for i in range(6):
    label = int(sample_labels[i])

    axes[0, i].imshow(sample_images[i])
    axes[0, i].set_title(f"Original: {class_names[label]}", fontsize=8)
    axes[0, i].axis('off')
    

    axes[1, i].imshow(augmented_images[i])
    axes[1, i].set_title(f"Augmented: {class_names[label]}", fontsize=8)
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# TODO: Build, compile, train, and evaluate a CNN that uses `data_augmentation`.
# Requirements:
# - start with Input(shape=(32, 32, 3))
# - apply `data_augmentation` inside the model before the convolutional feature extractor
# - use a CNN architecture suitable for CIFAR-10 classification
# - output 10 class probabilities
# - compile, train on `train_ds`, validate on `validation_ds`, and evaluate on `test_ds`
#
# Store the final compiled model in the variable `model_aug`.
raise NotImplementedError("Build the augmented CIFAR-10 CNN and assign it to `model_aug`.")

## Larger Custom CNN

Now we train the larger custom CNN from the previous notebook on the directory-backed datasets. The architecture is the same, except the last convolution block gets a projection shortcut before the activation.

In [ ]:
from keras.layers import Activation, Add, BatchNormalization, Rescaling

# TODO: Build a stronger custom CNN for CIFAR-10.
# Ideas to try:
# - more convolutional blocks than the basic model
# - Dropout for more stable training and regularization
# - optional residual/skip connections
# - optional data augmentation and rescaling inside the model
# - compare the result against the basic CNN
#
# Store the final compiled model in the variable `model_large` and train/evaluate it.
raise NotImplementedError("Build the larger custom CIFAR-10 CNN and assign it to `model_large`.")

## Transfer Learning
We might be quicker than the VL with this, however its quite a powerful concept and it fits here. Transfer learning is the technique of using a *large* pretrained network (usually not by us) as a base model, and simply adapting the output layers to our needs. The weights of the model stump or base can be *frozen*. 

What does that mean? How does it help?

In [ ]:
# import the base model
base_model = keras.applications.EfficientNetV2B0(
    include_top=False,  
    weights="imagenet",
    input_shape=(32, 32, 3),  
    pooling=None
)

base_model.trainable = True
inputs = keras.Input(shape=(32, 32, 3))
x = data_augmentation(inputs)
x = base_model(x, training=True)  
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)  
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(10, activation='softmax')(x)
model_effnet = Model(inputs=inputs, outputs=outputs)
model_effnet.compile(
        optimizer=keras.optimizers.Adam(),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
model_effnet.summary()

In [ ]:
from utils import train_model, eval_classification

effnet_history = train_model(model_effnet, X_train, y_train, "effnetb0_cifar10", epochs=25)
eval_classification(model_effnet, X_test, y_test, class_names, "CIFAR-10 EfficientNetB0")